# **First Experiment**

___

In [1]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Achyu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Achyu\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Achyu\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
import mlflow
import pandas as pd
import numpy as np
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

c:\Users\Achyu\OneDrive\Desktop\Anaconda\envs\atlas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("data.csv")

In [4]:
df.head()

,review,sentiment
0,Every great gangster movie has under-currents ...,positive
1,"I just saw this film last night, and I have to...",positive
2,This film is mildly entertaining if one neglec...,negative
3,Quentin Tarantino's partner in crime Roger Ava...,negative
4,I sat through this on TV hoping because of the...,negative


## **Data preprocessing**

In [5]:
def lemmatization(text):
    """Lemmatize the text"""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

In [6]:
def remove_stop_words(text):
    """Remove the stop words from the text"""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

In [7]:
def removing_numbers(text):
    """Remove numbers from text"""
    text = ' '.join([char for char in text if not char.isdigit()])
    return text

In [8]:
def lower_case(text):
    """Convert text to lower case"""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

In [9]:
def remove_punctuations(text):
    """Reomving the punctuations"""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace(":", "")
    text = re.sub('\s+', ' ', text).strip()
    return text

In [10]:
def remove_url(text):
    """Removing the URLs in text"""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

In [11]:
def normalize_text(df):
    """Normalize the text data"""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(remove_punctuations)
        df['review'] = df['review'].apply(remove_url)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error occurred during text normalization: {e}")
        raise

In [12]:
df = normalize_text(df)
df.head()

,review,sentiment
0,e v e r y g r e a t g a n g s t e r m o v i e ...,positive
1,s a w f i l m l a s t n i g h t s a y l o v e ...,positive
2,f i l m m i l d l y e n t e r t a i n i n g o ...,negative
3,q u e n t i n t a r a n t i n o s p a r t n e ...,negative
4,s a t t v h o p i n g n a m e s w o u l d w o ...,negative


In [13]:
df['sentiment'].value_counts()

sentiment
negative    269
positive    231
Name: count, dtype: int64

In [14]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [15]:
df['sentiment'] = df['sentiment'].map({'positive': 1, "negative": 0})
df.head()

,review,sentiment
0,e v e r y g r e a t g a n g s t e r m o v i e ...,1
1,s a w f i l m l a s t n i g h t s a y l o v e ...,1
2,f i l m m i l d l y e n t e r t a i n i n g o ...,0
3,q u e n t i n t a r a n t i n o s p a r t n e ...,0
4,s a t t v h o p i n g n a m e s w o u l d w o ...,0


In [31]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [32]:
df['review'].head(10)

0    e v e r y g r e a t g a n g s t e r m o v i e ...
1    s a w f i l m l a s t n i g h t s a y l o v e ...
2    f i l m m i l d l y e n t e r t a i n i n g o ...
3    q u e n t i n t a r a n t i n o s p a r t n e ...
4    s a t t v h o p i n g n a m e s w o u l d w o ...
5    e v e r y t h i n g m o v i e w r o n g w r o ...
6    n o n o n o n o n o n o n o f i l m e x c u s ...
7    p o s s i b l y w o r s t f i l m w i t h i n ...
8    s e e p e o p l e g i v i n g f i l m n e g a ...
9    g o t s u b j e c t e d p i l e o n e w e d n ...
Name: review, dtype: object

In [33]:
# Remove empty documents
df = df[df['review'].str.strip() != '']
df = df[df['review'].str.len() > 0]
print(f"Remaining documents after filtering: {len(df)}")

Remaining documents after filtering: 500


In [ ]:
vectorizer = CountVectorizer(max_features=100)
# Remove any remaining empty or whitespace-only documents
df = df[df['review'].str.strip() != '']
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)